# 02 — Rentabilidad y comportamiento de compra (A2)
Pregunta central: ¿dónde gana y dónde pierde margen RetailVenture?
Bloques:
1. Margen por categoría, canal y segmento
2. Efecto real de los descuentos
3. CLV por segmento y cohortes
4. Estacionalidad
Fuente: data/clean/ (salida del notebook 01).

## Bloque 1 — Margen por categoría, canal y segmento
Margen bruto = monto_neto − cantidad × costo del producto.

In [1]:
import pandas as pd
from pathlib import Path

CLEAN = Path("../data/clean")
cli = pd.read_csv(CLEAN / "clientes.csv", parse_dates=["fecha_registro", "primera_compra", "ultima_compra", "fecha_alta_efectiva"])
tx = pd.read_csv(CLEAN / "transacciones.csv", parse_dates=["fecha"])
pr = pd.read_csv(CLEAN / "productos.csv")

t = (tx.merge(pr[["producto_id", "categoria", "costo"]], on="producto_id")
       .merge(cli[["cliente_id", "segmento"]], on="cliente_id"))
t["margen"] = t.monto_neto - t.cantidad * t.costo

def resumen(dim):
    r = t.groupby(dim).agg(ventas=("monto_neto", "sum"), margen=("margen", "sum"),
                           transacciones=("transaccion_id", "count"), descuento_prom=("descuento", "mean"))
    r["margen_pct"] = r.margen / r.ventas * 100
    r["pct_ventas"] = r.ventas / r.ventas.sum() * 100
    r["pct_margen"] = r.margen / r.margen.sum() * 100
    r["descuento_prom"] *= 100
    return r.sort_values("ventas", ascending=False).round(1)

print(f"Ventas: ${t.monto_neto.sum():,.0f} | Margen: ${t.margen.sum():,.0f} "
      f"({t.margen.sum() / t.monto_neto.sum():.1%}) | Transacciones a pérdida: {(t.margen < 0).sum()}")

Ventas: $17,392,987 | Margen: $4,310,329 (24.8%) | Transacciones a pérdida: 711


In [2]:
resumen("categoria")

,ventas,margen,transacciones,descuento_prom,margen_pct,pct_ventas,pct_margen
categoria,,,,,,,
Electrónica,7662506.1,1049301.2,8419,5.3,13.7,44.1,24.3
Hogar_y_Deco,4465865.0,1353038.1,8355,5.1,30.3,25.7,31.4
Deportes,2592709.1,907213.7,8311,5.4,35.0,14.9,21.0
Ropa_y_Calzado,1513396.7,597005.3,8311,5.1,39.4,8.7,13.9
Belleza_y_Salud,748631.6,309502.6,8287,5.2,41.3,4.3,7.2
Alimentos_y_Bebidas,409879.0,94268.1,8317,5.1,23.0,2.4,2.2


In [3]:
resumen("canal")

,ventas,margen,transacciones,descuento_prom,margen_pct,pct_ventas,pct_margen
canal,,,,,,,
tienda_fisica,7862504.3,2046091.6,22343,3.7,26.0,45.2,47.5
online,6547309.2,1491185.5,19220,7.5,22.8,37.6,34.6
app_movil,2983174.0,773051.9,8437,3.7,25.9,17.2,17.9


In [4]:
resumen("segmento")

,ventas,margen,transacciones,descuento_prom,margen_pct,pct_ventas,pct_margen
segmento,,,,,,,
Frecuente,8766759.1,2131971.0,25458,5.6,24.3,50.4,49.5
Premium,5280660.8,1345562.6,14907,4.2,25.5,30.4,31.2
Esporádico,2701501.9,674499.2,7792,5.7,25.0,15.5,15.6
Nuevo,326779.1,80590.2,962,5.5,24.7,1.9,1.9
Inactivo,317286.6,77706.0,881,5.8,24.5,1.8,1.8


**Hallazgos:**
- **Electrónica vende mucho y deja poco**: 44% de las ventas, pero solo 24% del margen.
  Hogar y Deco hace lo contrario: 26% de las ventas y 31% del margen.
- **El canal online descuenta el doble** que la tienda física y por eso tiene peor margen
  (23% contra 26%).
- **Por segmento, los márgenes son parejos (24–26%)**: la diferencia está en el volumen.
  Frecuente y Premium generan el 81% del margen.
- **711 transacciones se venden a pérdida.** Se analizan en el bloque 2.